In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor

In [2]:
taxiTreino_clima = pd.read_csv('C:/Users/jhrat/Desktop/Mestrado BCC/Tópicos em Engenharia de Software/Projeto/taxiTreino_clima.csv')

In [3]:
# Função para agrupar os dados por hora
def preparar_dados(df):
    # Garante que a coluna de data é datetime
    df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'])
    
    # Arredonda a hora
    df['data_hora'] = df['lpep_pickup_datetime'].dt.floor('H')
    
    # Agrupa por hora
    df_agrupado = df.groupby('data_hora').agg(
        demanda_corridas=('lpep_pickup_datetime', 'count'),
        temperatura=('temperatura [C°]', 'mean'),
        umidade=('umidade relativa [%]', 'mean'),
        precipitacao=('precipitação acumulada em 1 hora [mm]', 'mean'),
        vento_vel=('velocidade do vento [mph]', 'mean'),
        feriado_fds=('feriado/fim de semana', 'max')
    ).reset_index()
    
    # Criar variáveis de tempo
    df_agrupado['hora_do_dia'] = df_agrupado['data_hora'].dt.hour
    df_agrupado['dia_da_semana'] = df_agrupado['data_hora'].dt.dayofweek
    df_agrupado['mes'] = df_agrupado['data_hora'].dt.month
    
    # Remove qualquer linha que tenha ficado com dados nulos no clima
    df_agrupado = df_agrupado.dropna()
    
    return df_agrupado

In [4]:
treino_agg = preparar_dados(taxiTreino_clima)


features = [
    'temperatura', 'umidade', 'precipitacao', 'vento_vel', 
    'feriado_fds', 'hora_do_dia', 'dia_da_semana', 'mes'
]

X_train = treino_agg[features]
y_train = treino_agg['demanda_corridas']

In [ ]:
# MODELO BASE
modelo_base = XGBRegressor(
    random_state=15
)


# GRADE DE PARÂMETROS
param_grid = {
    'n_estimators': [400, 550, 700],
    'max_depth': [4, 6, 8],
    'subsample': [0.2, 0.3, 0.5],
    'min_child_weight': [3, 5, 7],
    'num_parallel_tree': [1, 2, 4],
    'learning_rate': [0.001, 0.01, 0.1]
}#'learning_rate': 0.01, 'max_depth': 6, 'min_child_weight': 7, 'n_estimators': 700, 'num_parallel_tree': 4, 'subsample': 0.2


# CONFIGURANDO O GRID SEARCH
grid_search = GridSearchCV(
    estimator=modelo_base,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=3,
    n_jobs=-1,
    verbose=3
)


# TREINAMENTO
grid_search.fit(X_train, y_train)


# RESULTADOS
print("\n=== BUSCA CONCLUÍDA ===")
print("A melhor combinação de parâmetros foi:")
print(grid_search.best_params_)

# Melhor modelo
melhor_modelo = grid_search.best_estimator_

Fitting 3 folds for each of 729 candidates, totalling 2187 fits

=== BUSCA CONCLUÍDA ===
A melhor combinação de parâmetros foi:
{'learning_rate': 0.01, 'max_depth': 6, 'min_child_weight': 7, 'n_estimators': 700, 'num_parallel_tree': 4, 'subsample': 0.2}
